# Activity 2: Trnaist Light Curve Fitting

In this notebook, we will fit a real light curve containing an exoplanetary transit and occultation. Similar to Activity 1 (fitting RVs), the goal of this activity is to:
- get a direct contact with real data
- extract orbital and planetary characteristics from fitted parameters
- understand various challenges when fitting RV data

A specificity of the transit method is the choice of the physical model to fit. Data of excellent quality can be fitted with a very detailed model, which allows to extract more physics. Here, we will limit ourselves to the trapezoidal model, but will discuss the quality of this model.

In [ ]:
import numpy as np
import pandas as pd
import astropy.constants as ac
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## Step 1: Get the data

Go to the NASA Exoplanet Archive (https://exoplanetarchive.ipac.caltech.edu/) and download the photometric light curves of GJ 436 taken with IRAC4/Spitzer: UID_0057087_PLC_019.tbl (445 points) and UID_0057087_PLC_021.tbl (669 points). For now, let's work with the dataset 19 (first file).

In the cell below, load the data into a pandas dataframe, print it, and plot the photometric light curve series with uncertainty.

In [ ]:
# Your code here

lc_data = pd.read_csv()

# ---- Print ----
print(lc_data)

# ---- Plot ----
plt.errorbar()
plt.xlabel("Time [days]")
plt.ylabel("Relative flux [1]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(10,4)
plt.show()

## Step 2: Evaluate parameters

The first step is to visually inspect the data and evaluate the properties of the transit. 

________________________
Your answer here:<br><br>

The transit is clearly visible, there is no doubt where it is located, the fit will be robust.

The transit depth is about 0.8%.

We can clearly measure the duration of ingress and egress.

________________________

## Step 3: Define the model to fit

We are going to use `scipy.optimize.curve_fit`, which requires to define functions as $f(x,p_1,p_2,...)$ where $x$ is the variable, and $p_i$ the parameters.

In [ ]:
# The code below is based code provided by ChatGPT.
# It has been thoroughly reviewed, modified, and tested.
# Overall, it saved a bit of time, but didn't provide a "ready-to-go" model.

def trapezoid_transit_with_baseline(t, t0, depth, T_tot, T_full, c0=1.0, c1=0.0, tref=None):
    """
    Trapezoidal transit model with linear baseline.

    Model:
      F(t) = [1 - depth * s(t)] * (c0 + c1*(t - tref))

    where s(t) is:
      0 out of transit
      ramps linearly from 0->1 during ingress
      1 during full transit
      ramps linearly from 1->0 during egress

    Parameters
    ----------
    t : array-like (time)
    t0 : mid-transit time
    depth : transit depth (fractional drop, e.g. 0.01)
    T_tot : total transit duration (1st to 4th contact), same units as t
    T_full : full-transit duration (2nd to 3rd contact), same units as t
    c0, c1 : baseline parameters
    tref : reference time for baseline (default: median of t)
    """
    t = np.asarray(t, dtype=float)
    if tref is None:
        tref = np.median(t)

    # Ingress/egress duration
    tau = 0.5 * (T_tot - T_full)  # = t2 - t1 = t4 - t3

    # Contact times relative to t0
    t1 = t0 - T_full*0.5 - tau
    t2 = t0 - T_full*0.5
    t3 = t0 + T_full*0.5
    t4 = t0 + T_full*0.5 + tau

    # Shape function s(t) in [0,1]
    s = np.zeros_like(t)

    # Full transit
    in_full = (t >= t2) & (t <= t3)
    s[in_full] = 1.0

    # Shape function s(t) in [0,1]
    s = np.zeros_like(t)

    # Full transit
    in_full = (t >= t2) & (t <= t3)
    s[in_full] = 1.0

    # Ingress: t1 -> t2
    in_ing = (t >= t1) & (t < t2)
    s[in_ing] = (t[in_ing] - t1) / tau

    # Egress: t3 -> t4
    in_egr = (t > t3) & (t <= t4)
    s[in_egr] = (t4 - t[in_egr]) / tau

    transit = 1.0 - depth * s
    baseline = c0 + c1 * (t - tref)
    return transit * baseline



## Step 3 bis: visualize the model

Familiarizez yourself with the model and its parameters by plotting it for some parameters

In [ ]:
# Nothing to do here

time = np.linspace(0,10,500)
lc_series = trapezoid_transit_with_baseline(time, t0=4.0, depth=0.01, T_tot=0.5, T_full=0.4, c0=0.99, c1=0.01, tref=None)

# plotting a RV curve with default parameters
plt.plot(time, lc_series)
plt.xlabel("Time [days]")
plt.ylabel("Relative flux [1]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(10,4)
plt.show()

## Step 4: Fit the model to the data

In [ ]:
# Your code here

# -------------------------------------------------------
# 1) Initial guess
# -------------------------------------------------------

t0_init =  #days
depth_init =  #percent
T_tot_init =  #days
T_full_init =  #days
c0_init =  #baseline position
c1_init =  #baseline slope

p_init = [
    t0_init,
    depth_init,
    T_tot_init,
    T_full_init,
    c0_init,
    c1_init
]

# -------------------------------------------------------
# 3) Bounds (lower, upper)
# -------------------------------------------------------
lower_bounds = [
    ,
    ,
    ,
    ,
    ,
    
]

upper_bounds = [
    ,
    ,
    ,
    ,
    ,
    
]

# -------------------------------------------------------
# 4) Fit
# -------------------------------------------------------

x = 
y = 
yerr = 

param_opt, pcov = curve_fit(
    trapezoid_transit_with_baseline,
    x,
    y,
    p0=p_init,
    bounds=(lower_bounds, upper_bounds),
    sigma=yerr,
    absolute_sigma=True
)

param_err = np.sqrt(np.diag(pcov))

# -------------------------------------------------------
# 5) Print comparison table
# -------------------------------------------------------
param_names = ["t0", "depth", "T_tot", "T_full", "c0", "c1"]

param_df = pd.DataFrame({
    "Parameter": param_names,
    "Initial guess": p_init,
    "Best fit": param_opt,
    "1σ uncertainty": param_err
})

display(param_df)

## Step 5: Validate your fit visually

Plot the data and the fit to visually validate that the fit worked as intended.

In [ ]:
# Your code here

# generate the best fit model on a nice uniform grid
lc_duration = np.ptp()
time = np.linspace(0,lc_duration,500)
lc_model_fit = trapezoid_transit_with_baseline(time, *param_opt)

# residuals computed at data points
residuals = y - trapezoid_transit_with_baseline(x, *param_opt)

# ---- LC ----
plt.plot(time,lc_model_fit)

plt.errorbar(x,y,yerr=yerr,marker="+",c="orange",ls="")

plt.xlabel("Time [days]")
plt.ylabel("Relative flux [1]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(10,4)
plt.show()

# ---- Residuals ----
plt.errorbar(x,residuals,yerr=yerr,marker="+",c="orange",ls="")
plt.xlabel("Time [days]")
plt.ylabel("Residuals [1]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(10,4)
plt.show()

## Step 6: Red noise

Our fit is excellent, and the method is robust. However, it is still important to know what tools we have to assess the quality of a fit.<br>

A first quick check is to compute the root mean square (RMS) of the residuals and compare it to the median uncertainty $\sigma$. If RMS > $\sigma$, then the residuals contain more noise than signal. Compute both quantities and compare them.

In [ ]:
# Your code here

rms = ... residuals
sigma = np.median(yerr)
print(f"RMS   = {rms}")
print(f"sigma = {sigma}")

Another diagnostic tool is the computation of the red noise. In time-series photometry, red noise (or time-correlated noise) refers to residual variations that are not independent from one data point to the next. Unlike white noise, which is random and decreases as $1/\sqrt{N}$ when data are averaged, red noise introduces correlations over characteristic timescales (e.g., minutes to hours), often caused by atmospheric effects, instrumental drift, pointing variations, or imperfect baseline modeling. Because adjacent points are partially redundant, averaging does not reduce the noise as efficiently, leading to underestimated parameter uncertainties if one assumes purely white noise. In transit analysis, red noise typically appears as residual structure near ingress/egress or slow trends across the light curve. It can be diagnosed through the autocorrelation function or by checking whether the RMS of binned residuals follows the expected $N^{-1/2}$ scaling; deviations from this behavior indicate correlated noise and the need to inflate uncertainties or adopt a more sophisticated noise model.

During this process, we compute the $\beta$ factor, which measures how much correlated (red) noise is present in the residuals compared to ideal white noise. In general:
- $\beta\simeq1$ indicates a good fit
- $\beta>1.2$ indicates mild red noise
- $\beta>1.5$ significant correlated noise

In [ ]:
# nothing to do below

def red_noise_analysis(t_data,r_data):
    '''
    t_data : time
    r_data : residuals
    '''

    # Convert to numpy arrays
    r = np.asarray(r_data)
    t = np.asarray(t_data)
    
    # Remove mean
    r = r - np.mean(r)
    
    N = len(r)
    rms_unbinned = np.std(r, ddof=1)
    
    # --------------------------------------------------
    # 1) Autocorrelation Function (ACF)
    # --------------------------------------------------
    acf = np.correlate(r, r, mode='full')
    acf = acf[acf.size // 2:]          # keep positive lags
    acf /= acf[0]                      # normalize
    
    lags = np.arange(len(acf))
    
    # White-noise confidence envelope (~1/sqrt(N))
    conf = 1.0 / np.sqrt(N)
    
    # --------------------------------------------------
    # 2) Binned RMS and beta factor
    # --------------------------------------------------
    max_bin = min(200, N // 5)
    bin_sizes = np.arange(1, max_bin)
    
    rms_binned = []
    rms_white  = []
    beta       = []
    
    for m in bin_sizes:
        nbins = N // m
        if nbins < 2:
            break
        r_trim = r[:nbins * m]
        r_bin = r_trim.reshape(nbins, m).mean(axis=1)
        
        rms_m = np.std(r_bin, ddof=1)
        rms_binned.append(rms_m)
        
        # expected white-noise scaling
        white = rms_unbinned / np.sqrt(m)
        rms_white.append(white)
        
        beta.append(rms_m / white)
    
    rms_binned = np.array(rms_binned)
    rms_white  = np.array(rms_white)
    beta       = np.array(beta)
    
    beta_max = np.max(beta)
    
    # --------------------------------------------------
    # 3) Plot diagnostics
    # --------------------------------------------------
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    
    # ACF
    ax[0].plot(lags, acf)
    ax[0].axhline(conf, color='r', ls='--', alpha=0.5)
    ax[0].axhline(-conf, color='r', ls='--', alpha=0.5)
    ax[0].set_title("Autocorrelation of residuals")
    ax[0].set_xlabel("Lag")
    ax[0].set_ylabel("ACF")
    
    # Binned RMS
    ax[1].loglog(bin_sizes[:len(rms_binned)], rms_binned, label="Measured")
    ax[1].loglog(bin_sizes[:len(rms_white)], rms_white, ls='--', label="White noise")
    ax[1].set_xlabel("Bin size")
    ax[1].set_ylabel("RMS")
    ax[1].set_title(f"Binned RMS (beta_max ≈ {beta_max:.2f})")
    ax[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"Unbinned RMS: {rms_unbinned:.5g}")
    print(f"Max beta factor: {beta_max:.3f}")

In [ ]:
# Your code here
# add the time array
red_noise_analysis(,residuals)

# Repeating the procedure for the occultation

Let's repeat all the steps for the occultation. Apply the same steps to the data set 21 (second file):
- load and plot the data
- fit the lightcurve
- compute the red noise

In [ ]:
# Your code here

lc_data2 = pd.read_csv()

# ---- Plot ----
plt.errorbar()

plt.xlabel("Time [days]")
plt.ylabel("Relative flux [1]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(10,4)
plt.show()

In [ ]:
# Your code here

t0_occ_init =  #days
depth_occ_init =  #percent
T_tot_occ_init =  #days
T_full_occ_init =  #days
c0_occ_init =  #baseline position
c1_occ_init =  #baseline slope

p_occ_init = [
    t0_occ_init,
    depth_occ_init,
    T_tot_occ_init,
    T_full_occ_init,
    c0_occ_init,
    c1_occ_init
]

x2 = 
y2 = 
yerr2 = 

param_occ_opt, pcov_occ = curve_fit(
    trapezoid_transit_with_baseline,
    x2,
    y2,
    p0=p_occ_init,
    sigma=yerr2,
    absolute_sigma=True
)

param_occ_err = np.sqrt(np.diag(pcov_occ))

# -------------------------------------------------------
# 5) Print comparison table
# -------------------------------------------------------
param_occ_df = pd.DataFrame({
    "Parameter": param_names,
    "Initial guess": p_occ_init,
    "Best fit": param_occ_opt,
    "1σ uncertainty": param_occ_err
})

display(param_occ_df)

In [ ]:
# Your code here

# generate the best fit model on a nice uniform grid
lc_duration2 = np.ptp()
time2 = np.linspace(0,lc_duration2,500)
lc_model_fit2 = trapezoid_transit_with_baseline(time2, *param_occ_opt)

# residuals computed at data points
residuals2 = y2 - trapezoid_transit_with_baseline(x2, *param_occ_opt)

# ---- LC ----
plt.plot(time2,lc_model_fit2)

plt.errorbar(x2,y2,yerr=yerr2,marker="+",c="orange",ls="")

plt.xlabel("Time [days]")
plt.ylabel("Relative flux [1]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(10,4)
plt.show()

# ---- Residuals ----
plt.errorbar(x2,residuals2,yerr=yerr2,marker="+",c="orange",ls="")
plt.xlabel("Time [days]")
plt.ylabel("Residuals [1]")
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig = plt.gcf()
fig.set_size_inches(10,4)
plt.show()

In [ ]:
# Nothing to do here
red_noise_analysis(,residuals2)

## Physics time! -- computing the parameters of GJ 436 b

The fit confirms the existence of the exoplanet GJ 436 b!<br>

Using the transit depth and difference in ingress/egress time, compute the radius $R_\mathrm{p}$ of GJ 436 b with the associated uncertainty.<br>

Prograpation of uncertainties:
- if $f = kA^a B^b$, then $\sigma_f \approx \left| f \right| \sqrt{ \left(a\frac{\sigma_A}{A}\right)^2 + \left(b\frac{\sigma_B}{B}\right)^2 }$
- if $f = \sqrt{1-A^2}$, then $\sigma_f = \left| \frac{\sigma_A A}{f} \right|$

In [ ]:
# Your code below

# Formula to get Rp
def calc_Rp():
    
    return Rp 

# Formula to get uncertainty on Rp
def calc_sigma_Rp():
    
    return dRp

depth_opt = param_opt[1]
tau_ing_opt = param_opt[2]
tau_egr_opt = param_opt[3]
Rstar =  # Rsun

ddepth_opt = param_err[1]
dtau_ing_opt = param_err[2]
dtau_egr_opt = param_err[3]
dRstar =  # Rsun

Rp = calc_Rp()
dRp = calc_sigma_Rp()

print(f"The radius of GJ 436 b is Rp = {Rp:.3f} ± {dRp:.3f} R_Earth. It's a Neptune-size planet.")

Now let's use the occultation data to estimate the eccentricity $e$. Given:
- $t_\mathrm{tra}-t_\mathrm{occ} \simeq \frac{P}{2}\left[ 1 + \frac{4e\cos \omega}{\pi}\right]$
- $T_\mathrm{tra}/T_\mathrm{occ} \simeq 1 + e\sin \omega$

Compute $e\cos \omega$ and $e\sin \omega$, then use these value to obtain the eccentricity $e$ with uncertainty.

Note: if $f = \sqrt{aA^2 + bB^2}$ then $\sigma_f \approx \sqrt{\left(\frac{A}{f}\right)^2 a^2\sigma_A^2 + \left(\frac{B}{f}\right)^2 b^2\sigma_B^2 }$

In [ ]:
# Your code here

def calc_ecosw():
    
    return ecosw

def calc_sigma_ecosw():
    
    return sigma_ecosw

def calc_esinw():
    
    return esinw

def calc_sigma_esinw():
    
    return sigma_esinw

def calc_e():

    return e

def calc_sigma_e():
    
    return de

toffset = #not forgetting that we fitted t0_tra with a different origin in time
toffset2 = #not forgetting that we fitted t0_occ with a different origin in time
ttra = param_opt[0]+t_offset 
tocc = param_occ_opt[0]+t_offset2
Ttra = param_opt[2]
Tocc = param_occ_opt[2]
P =  # days

dttra = param_err[0]
dtocc = param_occ_err[0]
dTtra = param_err[2]
dTocc = param_occ_err[2]
dP =  # days

ecosw = calc_ecosw()
esinw = calc_esinw()
decosw = calc_sigma_ecosw()
desinw = calc_sigma_esinw()

print(f"e cos w = {ecosw:.5f}±{decosw:.5f}")
print(f"e sin w = {esinw:.5f}±{desinw:.5f}")
print()

e = calc_e(ecosw,esinw)
de = calc_sigma_e(ecosw,esinw,decosw,desinw)
print(f"The eccentricity of GJ 436 b is e = {e:.3f}±{de:.3f}")

## Conclusion

You now know how to fit a transit and extract valuable physical properties of the planet. We showed that even a simple trapezoidal model can reproduce a transit and an occultation very well.

A more complex model becomes very non-analytical very quickly. For this reason, there are many packages that exist to do it for you:
- juliet: https://juliet.readthedocs.io/en/latest/index.html)
- batman: https://lkreidberg.github.io/batman/docs/html/index.html
- starry: https://starry.readthedocs.io/en/latest/

The second reason why open source packages are preferred is because the parameters of a transit are diffi